# Laboratorio integral: suavizamiento exponencial, SES, Holt y Holt-Winters

**Objetivo.** Comprender la actualización de estados, el papel de los parámetros de suavizamiento, la estimación automática y el pronóstico de series temporales mediante métodos exponenciales.

**Recorrido:** 1. señal y filtro SES; 2. estimación de α; 3. nivel versus pronóstico ajustado; 4. Holt y tendencia; 5. Holt-Winters y estacionalidad; 6. tendencia amortiguada; 7. evaluación temporal; 8. experimentos guiados.

Este cuaderno es **autocontenido**: utiliza series sintéticas reproducibles y no requiere descargar datos. La intención es observar primero el comportamiento de los modelos cuando conocemos los componentes verdaderos de la señal.

## 0. Preparación

Instala las dependencias si es necesario (descomenta la siguiente celda). Se utiliza `statsmodels.tsa.holtwinters`, `pandas`, `numpy` y `matplotlib`.

In [ ]:
# %pip install -U numpy pandas matplotlib statsmodels scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from IPython.display import display

plt.rcParams.update({'figure.figsize': (12, 4.5), 'axes.grid': True, 'grid.alpha': 0.25})
rng = np.random.default_rng(2026)
print('Laboratorio listo')

## 1. La idea de SES: estado filtrado y pronóstico

El suavizamiento exponencial simple (SES) actualiza un **nivel** después de observar $y_t$:

$$\ell_t=\alpha y_t+(1-\alpha)\ell_{t-1},\qquad 0\leq\alpha\leq1.$$

Antes de observar $y_t$, el pronóstico de un paso es $\hat y_{t|t-1}=\ell_{t-1}$. Después de observarlo, el nivel pasa a $\ell_t$. Para horizontes futuros, SES pronostica un nivel constante:

$$\hat y_{t+h|t}=\ell_t,\quad h\geq1.$$

**Distinción clave:** `resultado.level` es el estado *posterior* a observar cada dato; `resultado.fittedvalues` es el pronóstico *previo* de un paso. Por ello, no deben confundirse al evaluar el ajuste.

### 1.1 Serie sintética de nivel casi constante

Generamos una señal con nivel real 50 y ruido independiente. El nivel verdadero se conoce y permite comparar el error del filtro.

In [ ]:
n_obs = 180
idx = pd.date_range('2020-01-01', periods=n_obs, freq='D')
nivel_verdadero = pd.Series(50.0, index=idx)
y_nivel = pd.Series(50 + rng.normal(0, 5, n_obs), index=idx, name='observada')
ax = y_nivel.plot(label='Observaciones', alpha=.6)
nivel_verdadero.plot(ax=ax, label='Nivel verdadero', linewidth=2)
ax.set_title('Señal sintética: nivel constante + ruido')
ax.legend(); plt.show()

### 1.2 Implementación manual del filtro

La siguiente función reproduce la recursión de SES. En este primer experimento fijamos el estado inicial para aislar el efecto de $\alpha$.

In [ ]:
def ses_manual(y, alpha, l0=None):
    y = pd.Series(y).astype(float)
    anterior = float(y.iloc[0] if l0 is None else l0)
    niveles, pred_un_paso = [], []
    for valor in y:
        pred_un_paso.append(anterior)
        anterior = alpha * valor + (1 - alpha) * anterior
        niveles.append(anterior)
    return (pd.Series(niveles, index=y.index, name='nivel'),
            pd.Series(pred_un_paso, index=y.index, name='pred_1_paso'))

fig, ax = plt.subplots()
y_nivel.plot(ax=ax, color='black', alpha=.35, label='Observada')
for alpha in [0.05, 0.2, 0.5, 1.0]:
    niveles, _ = ses_manual(y_nivel, alpha, l0=50)
    niveles.plot(ax=ax, label=f'α={alpha}')
ax.set_title('El parámetro α controla la velocidad de respuesta')
ax.legend(ncol=3); plt.show()

### 1.3 Interpretación como filtro IIR

Con el operador de retardo $L$, el filtro cumple:

$$\ell_t-(1-\alpha)\ell_{t-1}=\alpha y_t,\qquad H(L)=\frac{\alpha}{1-(1-\alpha)L}.$$

El polo es $1-\alpha$. Para $\alpha$ pequeño, el polo se aproxima a 1 y aumenta la memoria; para $\alpha=1$, $\ell_t=y_t$ y no hay suavizamiento. La respuesta al impulso para $k\geq0$ es $h[k]=\alpha(1-\alpha)^k$.

In [ ]:
k = np.arange(35)
fig, ax = plt.subplots()
for alpha in [.05, .2, .5, 1.0]:
    ax.plot(k, alpha*(1-alpha)**k, marker='.', label=f'α={alpha}')
ax.set_title('Respuesta al impulso del filtro SES')
ax.set_xlabel('Retardo k'); ax.set_ylabel('Peso de y[t-k]')
ax.legend(); plt.show()

## 2. ¿Por qué la optimización puede escoger $\alpha\approx1$?

En `SimpleExpSmoothing(...).fit(optimized=True)`, la implementación busca parámetros que minimicen el error de los **pronósticos de un paso**, no la rugosidad visual del nivel. Aproximadamente:

$$\min_{\alpha,\ell_0}\sum_t(y_t-\hat y_{t|t-1})^2.$$

Si la serie es muy persistente o se comporta como un paseo aleatorio, la última observación puede ser el mejor predictor de la siguiente. En ese caso SES puede escoger $\alpha\approx1$ aunque el nivel filtrado se parezca a la señal original.

In [ ]:
# Dos regímenes: ruido alrededor de nivel fijo vs paseo aleatorio
rng2 = np.random.default_rng(17)
idx2 = pd.date_range('2022-01-01', periods=240, freq='D')
serie_estable = pd.Series(100 + rng2.normal(0, 4, 240), index=idx2)
serie_rw = pd.Series(100 + np.cumsum(rng2.normal(0, 4, 240)), index=idx2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, serie, nombre in zip(axes, [serie_estable, serie_rw], ['Nivel fijo + ruido', 'Paseo aleatorio']):
    ajuste = SimpleExpSmoothing(serie, initialization_method='estimated').fit(optimized=True)
    serie.plot(ax=ax, label='Observada', alpha=.55)
    ajuste.level.plot(ax=ax, label='Nivel estimado', linewidth=2)
    ax.set_title(f'{nombre}: α={ajuste.params["smoothing_level"]:.3f}')
    ax.legend()
plt.tight_layout(); plt.show()

### 2.1 Comparar la función objetivo para distintos α

Para una comparación controlada, mantenemos fijo el estado inicial al calcular la suma de errores cuadrados. La optimización completa de `statsmodels` puede variar también los estados iniciales; por ello, los mínimos no tienen que coincidir exactamente.

In [ ]:
alfas = np.linspace(.01, 1, 100)
fig, ax = plt.subplots()
for serie, nombre in [(serie_estable, 'Nivel fijo'), (serie_rw, 'Paseo aleatorio')]:
    valores = []
    for a in alfas:
        _, pred = ses_manual(serie, a, l0=float(serie.iloc[0]))
        valores.append(np.sum((serie.iloc[1:] - pred.iloc[1:])**2))
    ax.plot(alfas, np.array(valores)/len(serie.iloc[1:]), label=nombre)
ax.set_title('Error cuadrático de predicción de un paso frente a α')
ax.set_xlabel('α'); ax.set_ylabel('ECM'); ax.legend(); plt.show()

## 3. Ajuste SES: estados, valores ajustados y pronóstico

En la serie con paseo aleatorio separamos entrenamiento y prueba. La predicción de SES para todos los horizontes posteriores al entrenamiento es constante. El intervalo de prueba no interviene en el ajuste.

In [ ]:
h = 30
train, test = serie_rw.iloc[:-h], serie_rw.iloc[-h:]
ses = SimpleExpSmoothing(train, initialization_method='estimated').fit(optimized=True)
pred_ses = ses.forecast(h)

fig, ax = plt.subplots(figsize=(13, 5))
train.plot(ax=ax, label='Entrenamiento', alpha=.5)
ses.level.plot(ax=ax, label='Nivel filtrado ℓₜ', linewidth=2)
ses.fittedvalues.plot(ax=ax, label='Pronóstico histórico de un paso', alpha=.65)
test.plot(ax=ax, label='Prueba', color='black')
pred_ses.plot(ax=ax, label='Pronóstico futuro SES', linestyle='--', linewidth=2)
ax.set_title('Nivel ≠ predicción histórica de un paso ≠ pronóstico futuro')
ax.legend(ncol=2); plt.show()
print('Parámetros SES:')
print(ses.params)

## 4. Métricas y comparación con pronóstico ingenuo

El pronóstico ingenuo (*naive*) repite la última observación: $\hat y_{T+h|T}=y_T$. Para SES con $\alpha=1$, ambos coinciden en su predicción puntual. Evaluaremos MAE y RMSE en un período que el modelo no vio durante el ajuste.

In [ ]:
def metricas(y_real, y_pred):
    a = np.asarray(y_real, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    return {'MAE': np.mean(np.abs(a-p)), 'RMSE': np.sqrt(np.mean((a-p)**2))}

naive = pd.Series(float(train.iloc[-1]), index=test.index)
print(pd.DataFrame({'SES': metricas(test, pred_ses), 'Naive': metricas(test, naive)}).T.round(3))

## 5. Holt: añadir una tendencia

Cuando el nivel crece o decrece, SES no modela la pendiente. Holt incorpora dos estados:

$$\ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+b_{t-1})$$
$$b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}.$$

Su pronóstico sin amortiguación es:

$$\boxed{\hat y_{t+h|t}=\ell_t+h b_t}.$$

**Atención:** `b_t` es el incremento estimado por período, no un coeficiente de regresión fijo. Tanto nivel como pendiente se actualizan recursivamente.

In [ ]:
rng3 = np.random.default_rng(81)
n = 180
t = np.arange(n)
idx3 = pd.date_range('2021-01-01', periods=n, freq='D')
trend_true = 25 + .35*t
y_trend = pd.Series(trend_true + rng3.normal(0, 3, n), index=idx3)
train_h, test_h = y_trend.iloc[:-24], y_trend.iloc[-24:]

ses_h = SimpleExpSmoothing(train_h, initialization_method='estimated').fit(optimized=True)
holt = Holt(train_h, initialization_method='estimated').fit(optimized=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
train_h.plot(ax=axes[0], alpha=.45, label='Observada')
pd.Series(trend_true[:len(train_h)], index=train_h.index).plot(ax=axes[0], label='Tendencia verdadera')
holt.level.plot(ax=axes[0], label='Nivel Holt')
axes[0].legend(); axes[0].set_title('Nivel estimado')
holt.trend.plot(ax=axes[1], label='Pendiente estimada bₜ')
axes[1].axhline(.35, color='black', linestyle='--', label='Pendiente verdadera 0.35')
axes[1].legend(); axes[1].set_title('Estado de tendencia')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots()
train_h.plot(ax=ax, alpha=.45, label='Train')
test_h.plot(ax=ax, color='black', label='Test')
ses_h.forecast(len(test_h)).plot(ax=ax, label='SES', linestyle='--')
holt.forecast(len(test_h)).plot(ax=ax, label='Holt', linestyle='--')
ax.legend(); ax.set_title('La tendencia cambia el pronóstico de varios pasos'); plt.show()
print(pd.DataFrame({'SES': metricas(test_h, ses_h.forecast(len(test_h))),
                    'Holt': metricas(test_h, holt.forecast(len(test_h)))}).T.round(3))

## 6. Holt-Winters: nivel + tendencia + estacionalidad

En el modelo **aditivo**, con período estacional $m$:

$$\ell_t=\alpha(y_t-s_{t-m})+(1-\alpha)(\ell_{t-1}+b_{t-1})$$
$$b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}$$
$$s_t=\gamma(y_t-\ell_{t-1}-b_{t-1})+(1-\gamma)s_{t-m}.$$

El pronóstico a $h$ pasos es:

$$\boxed{\hat y_{t+h|t}=\ell_t+h b_t+s_{t+h-m(k+1)},\quad k=\lfloor(h-1)/m\rfloor.}$$

El subíndice estacional reutiliza la última estimación disponible para la misma posición del ciclo. Con estacionalidad multiplicativa, se reemplaza la suma por el producto: $(\ell_t+hb_t)s_{t+h-m(k+1)}$.

### 6.1 Construcción de una serie mensual con componentes conocidos

La amplitud estacional es constante, por lo que corresponde probar un modelo aditivo. El patrón mensual suma cero y se repite cada 12 meses.

In [ ]:
rng4 = np.random.default_rng(42)
m = 12
n = 120
fechas = pd.date_range('2015-01-01', periods=n, freq='MS')
t = np.arange(n)
patron = np.array([-15,-12,-5,0,8,15,20,18,10,0,-7,-12], dtype=float)
patron = patron - patron.mean()
nivel_real = 100 + .55*t
estacional_real = np.tile(patron, n//m)
y_hw = pd.Series(nivel_real + estacional_real + rng4.normal(0, 3, n), index=fechas)

fig, ax = plt.subplots()
y_hw.plot(ax=ax, label='Observada')
ax.plot(fechas, nivel_real, label='Nivel + tendencia reales', linewidth=2)
ax.set_title('Serie sintética mensual'); ax.legend(); plt.show()

In [ ]:
h_hw = 24
train_hw, test_hw = y_hw.iloc[:-h_hw], y_hw.iloc[-h_hw:]
hw = ExponentialSmoothing(train_hw, trend='add', seasonal='add',
                          seasonal_periods=12, initialization_method='estimated').fit(optimized=True)
print('Parámetros estimados:')
for nombre, valor in hw.params.items():
    if isinstance(valor, (int, float, np.number)):
        print(f'{nombre}: {valor:.5f}')

### 6.2 Inspeccionar los estados estimados

`hw.level`, `hw.trend` y `hw.season` contienen las trayectorias de los tres estados. Son **estimaciones** de las componentes, no observaciones directas; pueden diferir de los componentes verdaderos aunque el pronóstico sea adecuado.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
hw.level.plot(ax=axes[0], label='Nivel estimado')
axes[0].plot(train_hw.index, nivel_real[:len(train_hw)], '--', label='Nivel real')
axes[0].legend(); axes[0].set_title('Nivel')
hw.trend.plot(ax=axes[1], label='Tendencia estimada')
axes[1].axhline(.55, linestyle='--', color='black', label='Pendiente real')
axes[1].legend(); axes[1].set_title('Tendencia')
hw.season.plot(ax=axes[2], label='Estacionalidad estimada')
axes[2].plot(train_hw.index, estacional_real[:len(train_hw)], '--', alpha=.7, label='Estacionalidad real')
axes[2].legend(); axes[2].set_title('Estacionalidad')
plt.tight_layout(); plt.show()

### 6.3 Reconstrucción del pronóstico de Holt-Winters

La expresión $\ell_T+h b_T+s_{\text{mes futuro}}$ puede implementarse directamente a partir de los estados finales. En un modelo aditivo no amortiguado, debe reproducir el pronóstico de `statsmodels`, salvo pequeñas diferencias numéricas.

In [ ]:
def pronostico_hw_aditivo_manual(resultado, horizonte, periodo):
    nivel = float(resultado.level.iloc[-1])
    pendiente = float(resultado.trend.iloc[-1])
    ultimos_estacionales = np.asarray(resultado.season.iloc[-periodo:], dtype=float)
    valores = [nivel + j*pendiente + ultimos_estacionales[(j-1)%periodo]
               for j in range(1, horizonte+1)]
    return np.asarray(valores)

pred_hw = hw.forecast(h_hw)
pred_manual = pronostico_hw_aditivo_manual(hw, h_hw, 12)
print('Máxima diferencia entre pronósticos:', np.max(np.abs(np.asarray(pred_hw)-pred_manual)))

fig, ax = plt.subplots(figsize=(13, 5))
train_hw.plot(ax=ax, label='Entrenamiento', alpha=.65)
test_hw.plot(ax=ax, label='Prueba', color='black')
pred_hw.plot(ax=ax, label='Holt-Winters', linewidth=2)
ax.set_title('Pronóstico mensual: tendencia + estacionalidad'); ax.legend(); plt.show()

## 7. Comparación con SES y Holt en la serie estacional

Los modelos se entrenan sobre el mismo tramo y se comparan sobre las mismas 24 observaciones futuras. No se utiliza el conjunto de prueba para seleccionar parámetros.

In [ ]:
ses_est = SimpleExpSmoothing(train_hw, initialization_method='estimated').fit(optimized=True)
holt_est = Holt(train_hw, initialization_method='estimated').fit(optimized=True)
modelos = {'Naive': pd.Series(float(train_hw.iloc[-1]), index=test_hw.index),
           'SES': ses_est.forecast(h_hw),
           'Holt': holt_est.forecast(h_hw),
           'Holt-Winters': pred_hw}
comparacion = pd.DataFrame({nombre: metricas(test_hw, p) for nombre, p in modelos.items()}).T
print(comparacion.round(3))
ax = test_hw.plot(label='Real', color='black', linewidth=2)
for nombre, p in modelos.items():
    pd.Series(np.asarray(p), index=test_hw.index).plot(ax=ax, label=nombre, alpha=.8)
ax.set_title('Pronósticos fuera de muestra'); ax.legend(ncol=3); plt.show()

## 8. Estacionalidad multiplicativa

Cuando la amplitud de las oscilaciones crece con el nivel, puede convenir un factor estacional multiplicativo. El pronóstico es:

$$\hat y_{t+h|t}=(\ell_t+h b_t)s_{t+h-m(k+1)}.$$

La variante multiplicativa requiere observaciones estrictamente positivas. Compararemos ambos tipos sobre una señal positiva cuya amplitud estacional aumenta con el nivel.

In [ ]:
rng5 = np.random.default_rng(99)
n = 120
t = np.arange(n)
fechas2 = pd.date_range('2015-01-01', periods=n, freq='MS')
base = 80 + .9*t
factores = np.array([.8,.83,.9,1.0,1.08,1.16,1.22,1.17,1.09,1.0,.91,.84])
y_mult = pd.Series(base*np.tile(factores, n//12) + rng5.normal(0, 2, n), index=fechas2)
tr_m, te_m = y_mult.iloc[:-24], y_mult.iloc[-24:]
fit_add = ExponentialSmoothing(tr_m, trend='add', seasonal='add', seasonal_periods=12,
                               initialization_method='estimated').fit(optimized=True)
fit_mul = ExponentialSmoothing(tr_m, trend='add', seasonal='mul', seasonal_periods=12,
                               initialization_method='estimated').fit(optimized=True)
print(pd.DataFrame({'Aditivo': metricas(te_m, fit_add.forecast(24)),
                    'Multiplicativo': metricas(te_m, fit_mul.forecast(24))}).T.round(3))
ax = tr_m.plot(label='Train', alpha=.5)
te_m.plot(ax=ax, label='Test', color='black')
fit_add.forecast(24).plot(ax=ax, label='HW aditivo')
fit_mul.forecast(24).plot(ax=ax, label='HW multiplicativo')
ax.legend(); ax.set_title('Amplitud estacional proporcional al nivel'); plt.show()

## 9. Tendencia amortiguada

En Holt con tendencia amortiguada, la extrapolación utiliza $0<\phi<1$:

$$\hat y_{t+h|t}=\ell_t+\left(\sum_{j=1}^h\phi^j\right)b_t.$$

Si $\phi<1$, el incremento acumulado de la tendencia se estabiliza cuando $h$ crece. En Holt-Winters aditivo, se suma además el estado estacional correspondiente.

In [ ]:
holt_normal = Holt(train_h, initialization_method='estimated').fit(optimized=True)
holt_amort = Holt(train_h, damped_trend=True, initialization_method='estimated').fit(optimized=True)
future_h = 70
fig, ax = plt.subplots()
train_h.iloc[-70:].plot(ax=ax, label='Histórico reciente', color='black')
holt_normal.forecast(future_h).plot(ax=ax, label='Holt sin amortiguar')
holt_amort.forecast(future_h).plot(ax=ax, label='Holt amortiguado')
ax.set_title('Extrapolación de tendencia a largo plazo'); ax.legend(); plt.show()
print('Parámetro de amortiguación:', holt_amort.params['damping_trend'])

## 10. Diagnóstico de residuos

Una serie de residuos con autocorrelación apreciable sugiere que la estructura de estados no capturó toda la dependencia temporal. La prueba de Ljung-Box es una ayuda diagnóstica, no una garantía de calidad predictiva. Aquí inspeccionamos los residuos de pronóstico de un paso de Holt-Winters.

In [ ]:
residuos = train_hw - hw.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
residuos.plot(ax=axes[0], title='Residuos de un paso')
axes[1].hist(residuos.dropna(), bins=18)
axes[1].set_title('Distribución de residuos')
plt.tight_layout(); plt.show()
print(acorr_ljungbox(residuos.dropna(), lags=[6, 12], return_df=True))

## 11. Laboratorio guiado: preguntas para experimentar

1. **Suavizado versus pronóstico.** En la sección 2, cambia la desviación del ruido de la serie de nivel fijo de 4 a 12. ¿Qué ocurre con el $\alpha$ optimizado? ¿Por qué?
2. **Cambio de nivel.** Genera una serie constante durante 100 períodos y súmale 25 unidades a partir del período 101. Compara el tiempo de respuesta de SES para $\alpha=0.05,0.2,0.8$.
3. **Tendencia.** En la sección 5, sustituye la pendiente verdadera 0.35 por 0 y vuelve a comparar SES con Holt.
4. **Horizonte.** Evalúa el modelo Holt-Winters con horizontes 6, 12 y 24 meses, reentrenando cada vez con el mismo criterio de separación temporal. ¿Cómo cambia el error?
5. **Período estacional.** Ajusta erróneamente `seasonal_periods=6` a la serie mensual de período 12. ¿Qué observas en el pronóstico y los residuos?
6. **Tipo estacional.** Compara modelos aditivo y multiplicativo sobre las dos series estacionales. ¿En qué caso se justifica cada uno?
7. **Pronóstico amortiguado.** Compara Holt con y sin amortiguación para horizontes de 12 y 70 pasos. Explica qué supuesto de crecimiento incorpora cada uno.
8. **Identificación de estados.** ¿Por qué `level` y `fittedvalues` no son idénticos? Verifícalo en las primeras cinco observaciones.

**Entregable sugerido:** para cada experimento, incluye gráfica, parámetros estimados, MAE/RMSE y una explicación causal del comportamiento del método.

## 12. Síntesis conceptual

| Método | Estados | Pronóstico puntual | Uso típico |
|---|---|---|---|
| SES | Nivel $\ell_t$ | $\ell_t$ | Sin tendencia ni estacionalidad |
| Holt | Nivel y tendencia $\ell_t,b_t$ | $\ell_t+hb_t$ | Tendencia sin estacionalidad |
| Holt amortiguado | Nivel, tendencia y $\phi$ | $\ell_t+\sum_{j=1}^h\phi^j b_t$ | Tendencia que se atenúa |
| Holt-Winters aditivo | Nivel, tendencia, estacionalidad | $\ell_t+hb_t+s_{\text{fase}}$ | Amplitud estacional aproximadamente constante |
| Holt-Winters multiplicativo | Nivel, tendencia, factor estacional | $(\ell_t+hb_t)s_{\text{fase}}$ | Amplitud estacional proporcional al nivel |

**Conclusión:** estos métodos actualizan estados para producir pronósticos. Suavizar mucho y pronosticar bien no son objetivos equivalentes. Los parámetros optimizados dependen de la estructura de la serie y del criterio de error; deben contrastarse mediante evaluación fuera de muestra.

**Nota de alcance:** el cuaderno usa la implementación clásica de Holt-Winters de `statsmodels`. Una formulación ETS probabilística completa añade supuestos explícitos sobre innovaciones y permite tratar, entre otros aspectos, intervalos de predicción desde el modelo estadístico.